# 我的第一个实验室 = 我的第一个前沿法学硕士项目
## 总结所有没有 Selenium 的网站
这个简单的“应用程序”使用 Jina (https://jina.ai/reader) 将所有网站转换为 Markdown，然后由法学硕士进行总结。正如他们的网站所说：“只需在前面添加 r.jina.ai，即可将 URL 转换为 LLM 友好的输入”。他们还有其他看起来也很有用的工具。




In [ ]:
# 导入依赖
# imports

import os
import requests                                 # added for jina
from dotenv import load_dotenv
# from scraper import fetch_website_contents    # not needed for jina
from IPython.display import Markdown, display
from openai import OpenAI


In [ ]:
# Load environment variables from a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# 检查 API Key
# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

# Setup access to the frontier model

openai = OpenAI()

In [ ]:
# Step 1-a: Define the user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""

In [ ]:
# Step 1-b: Define the system prompt

system_prompt = """
You are a smart assistant that analyzes the contents of a website,
and provides a short, clear, summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [ ]:
# Add the website content to the user prompt

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [ ]:
# Step 5: Change the content utility to use jina

def fetch_url_content(url):
    jina_reader_url = f"https://r.jina.ai/{url}"
    try:
        response = requests.get(jina_reader_url)
        response.raise_for_status()                     # Raise an exception for HTTP errors
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching URL: {e}")
        return None


In [ ]:
# Step 3: Call OpenAI & Step 4: print the result

def summarize(url):
    website = fetch_url_content(url)
    response = openai.chat.completions.create(
        model = "gpt-5-nano",
        messages = messages_for(website)
    )
    summary = response.choices[0].message.content
    return display(Markdown(summary))


In [ ]:
summarize("https://edwarddonner.com")

In [ ]:
summarize("https://cnn.com")

In [ ]:
summarize("https://openai.com")

## 内容摘要与技术摘要

在我的工作中，一个网站或一组网站的技术摘要也很有用。例如，它是在服务器（HTML）还是在浏览器（JavaScript）中呈现，使用了什么内容管理系统（CMS），有多少页面，有多少出站链接，有多少入站链接等。通过这个练习，我意识到LLM可以帮助分析内容，但我可能需要其他工具来计算页面、链接和其他规范。

向在社区贡献中添加“Market_Research_Agent.ipynb”的人“大声喊叫”。这是使用法学硕士作为管理顾问的一个很好的例子。我认为 Jina 可能会通过 API 提供网络搜索结果来提供给您的法学硕士，从而帮助解决这个用例。这是该笔记本的系统提示，我计划经常使用这种格式。

system_prompt = """你要扮演专门从事市场研究的麦肯锡顾问。 
1）您应遵守法律准则，切勿提出不道德的建议。 
2) 您的工作是通过分析客户公司的计划并提出新计划的建议，为客户实现利润最大化。\n 
3）遵循行业回答框架，始终给出简单的答案并抓住重点。
4) 如果可能的话，尝试看看存在哪些竞争对手以及您的客户公司可以利用哪些市场空白。
5) 此外，使用SWOT、Porters 5强制总结您的建议，对每个建议给出置信度分数
6) 通过查看市场差距来尝试给出独特的解决方案，如果市场差距不明确，请跳过此步骤
7) 添加对公司收入增长速度的估计，前提是他们遵循指导方针，并在考虑到非理想条件的情况下给出保守的估计。
8) 如果该网站不是公司的或数据不可用，请给出错误消息，并需要更多数据进行分析"""